# OpenSky Network → Neo4j (Consultas Analíticas v2)


**Prerequisitos:**
- Haber ejecutado `cassandra_schema_migration.py`
- Haber ejecutado `load_airports.py`
- Pipeline corriendo al menos un ciclo completo (5 min de ingesta + Spark)

## 1. Configuración y conexión

In [ ]:
from neo4j import GraphDatabase
import pandas as pd

NEO4J_URI      = "bolt://localhost:7687"
NEO4J_USER     = "neo4j"
NEO4J_PASSWORD = "password"
NEO4J_DATABASE = "neo4j"
NEAR_RADIUS_KM = 50

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()

def run(cypher, **params):
    with driver.session(database=NEO4J_DATABASE) as s:
        return pd.DataFrame(s.run(cypher, **params).data())

print("Conectado a Neo4j.")

## 2. Verificación del grafo

In [ ]:
COUNT_QUERIES = {
    "Aircraft":        "MATCH (a:Aircraft) RETURN count(a) AS n",
    "Country":         "MATCH (c:Country) RETURN count(c) AS n",
    "Airport":         "MATCH (a:Airport) RETURN count(a) AS n",
    "Snapshot":        "MATCH (s:Snapshot) RETURN count(s) AS n",
    "OPERATES":        "MATCH ()-[r:OPERATES]->() RETURN count(r) AS n",
    "SNAPSHOT":        "MATCH ()-[r:SNAPSHOT]->() RETURN count(r) AS n",
    "NEAR":            "MATCH ()-[r:NEAR]-() RETURN count(r)/2 AS n",
    "DEPARTED_FROM":   "MATCH ()-[r:DEPARTED_FROM]->() RETURN count(r) AS n",
    "ARRIVED_AT":      "MATCH ()-[r:ARRIVED_AT]->() RETURN count(r) AS n",
}

with driver.session(database=NEO4J_DATABASE) as session:
    print(f"{'Entidad':<20} {'Cantidad':>10}")
    print("-" * 32)
    for label, query in COUNT_QUERIES.items():
        n = session.run(query).single()["n"]
        print(f"{label:<20} {n:>10,}")

## 3. Consultas originales

### Q1 – Top 10 países con más aeronaves

In [ ]:
df = run("""
    MATCH (c:Country)-[:OPERATES]->(a:Aircraft)
    RETURN c.name AS country, count(a) AS aircraft_count
    ORDER BY aircraft_count DESC LIMIT 10
""")
print("Top 10 países por aeronaves:"); print(df.to_string(index=False))

### Q2 – Aeronaves con mayor velocidad promedio

In [ ]:
df = run("""
    MATCH (a:Aircraft)-[r:SNAPSHOT]->(s:Snapshot)
    WHERE r.on_ground = false AND r.velocity IS NOT NULL
    WITH a, avg(r.velocity) AS avg_vel, count(r) AS snapshots
    WHERE snapshots >= 3
    RETURN a.icao24 AS icao24, a.callsign AS callsign,
           round(avg_vel, 2) AS avg_velocity_ms, snapshots
    ORDER BY avg_velocity_ms DESC LIMIT 10
""")
print("Top 10 por velocidad promedio:"); print(df.to_string(index=False))

### Q3 – Hub de proximidad (aeronaves con más vecinas a ≤50 km)

In [ ]:
df = run("""
    MATCH (a:Aircraft)-[r:NEAR]-(b:Aircraft)
    WITH a, count(DISTINCT b) AS near_count, avg(r.dist_km) AS avg_dist
    RETURN a.icao24 AS icao24, a.callsign AS callsign,
           near_count, round(avg_dist, 2) AS avg_dist_km
    ORDER BY near_count DESC LIMIT 10
""")
print(f"Aeronaves con más vecinas a ≤{NEAR_RADIUS_KM} km:"); print(df.to_string(index=False))

### Q4 – Trayectoria temporal de una aeronave

In [ ]:
with driver.session(database=NEO4J_DATABASE) as s:
    top = s.run(
        "MATCH (a:Aircraft)-[r:SNAPSHOT]->() "
        "RETURN a.icao24 AS icao24, count(r) AS n ORDER BY n DESC LIMIT 1"
    ).single()
    EXAMPLE_ICAO = top["icao24"]
    print(f"Aeronave: {EXAMPLE_ICAO} ({top['n']} snapshots)")

df = run("""
    MATCH (a:Aircraft {icao24: $icao24})-[r:SNAPSHOT]->()
    WHERE r.baro_altitude IS NOT NULL
    RETURN r.snapshot_time AS ts, r.baro_altitude AS alt_m,
           r.velocity AS vel_ms, r.velocity_kmh AS vel_kmh,
           r.vertical_rate AS vr_ms, r.latitude AS lat, r.longitude AS lon
    ORDER BY r.snapshot_time ASC
""", icao24=EXAMPLE_ICAO)
print(df[["ts", "alt_m", "vel_ms", "vel_kmh", "vr_ms"]].to_string(index=False))

### Q5 – Interacciones aéreas entre países

In [ ]:
df = run("""
    MATCH (c1:Country)-[:OPERATES]->(a1:Aircraft)-[:NEAR]-(a2:Aircraft)<-[:OPERATES]-(c2:Country)
    WHERE c1 <> c2
    WITH c1.name AS country_a, c2.name AS country_b, count(*) AS interactions
    RETURN country_a, country_b, interactions
    ORDER BY interactions DESC LIMIT 15
""")
print("Top 15 pares de países con mayor proximidad:"); print(df.to_string(index=False))

### Q6 – Distribución por fuente de posición

In [ ]:
df = run("""
    MATCH (a:Aircraft)
    RETURN a.position_source_label AS source_label, count(a) AS n
    ORDER BY n DESC
""")
print("Distribución por fuente de posición:"); print(df.to_string(index=False))

## 4. Consultas de hotspots y rutas

### Q7 – Hotspots de salida: aeropuertos con más despegues detectados

In [ ]:
df = run("""
    MATCH (a:Aircraft)-[r:DEPARTED_FROM]->(ap:Airport)
    WHERE r.confidence IN ['HIGH', 'MEDIUM']
    RETURN ap.icao          AS airport_icao,
           ap.iata          AS airport_iata,
           ap.name          AS airport_name,
           ap.city          AS city,
           ap.country       AS country,
           count(r)         AS departures,
           count(DISTINCT a) AS unique_aircraft
    ORDER BY departures DESC
    LIMIT 20
""")
print("Top 20 aeropuertos por despegues detectados (confianza HIGH/MEDIUM):")
print(df.to_string(index=False))

### Q8 – Hotspots de llegada: aeropuertos con más aterrizajes

In [ ]:
df = run("""
    MATCH (a:Aircraft)-[r:ARRIVED_AT]->(ap:Airport)
    WHERE r.confidence IN ['HIGH', 'MEDIUM']
    RETURN ap.icao          AS airport_icao,
           ap.iata          AS airport_iata,
           ap.name          AS airport_name,
           ap.city          AS city,
           ap.country       AS country,
           count(r)         AS arrivals,
           count(DISTINCT a) AS unique_aircraft
    ORDER BY arrivals DESC
    LIMIT 20
""")
print("Top 20 aeropuertos por aterrizajes detectados (confianza HIGH/MEDIUM):")
print(df.to_string(index=False))

### Q9 – Rutas más frecuentes: pares origen → destino

Detecta el patrón `(Aircraft)-[:DEPARTED_FROM]->(ap_origen)` y
`(mismo Aircraft)-[:ARRIVED_AT]->(ap_destino)` donde el despegue
ocurrió **antes** del aterrizaje.

In [ ]:
df = run("""
    MATCH (a:Aircraft)-[dep:DEPARTED_FROM]->(origin:Airport),
          (a)-[arr:ARRIVED_AT]->(dest:Airport)
    WHERE dep.event_time < arr.event_time
      AND dep.confidence IN ['HIGH', 'MEDIUM']
      AND arr.confidence IN ['HIGH', 'MEDIUM']
      AND origin <> dest
    WITH origin.icao   AS origin_icao,
         origin.iata   AS origin_iata,
         origin.city   AS origin_city,
         dest.icao     AS dest_icao,
         dest.iata     AS dest_iata,
         dest.city     AS dest_city,
         count(*)      AS flights
    RETURN origin_icao, origin_iata, origin_city,
           dest_icao,   dest_iata,   dest_city,
           flights
    ORDER BY flights DESC
    LIMIT 25
""")
print("Top 25 rutas más frecuentes:")
print(df.to_string(index=False))

### Q10 – Tráfico neto por aeropuerto (salidas − llegadas)

Un valor positivo = más salidas que llegadas (aeropuerto emisor).
Un valor negativo = más llegadas (aeropuerto receptor / hub de destino).

In [ ]:
df = run("""
    MATCH (ap:Airport)
    OPTIONAL MATCH (ap)<-[dep:DEPARTED_FROM]-(:Aircraft)
        WHERE dep.confidence IN ['HIGH', 'MEDIUM']
    OPTIONAL MATCH (ap)<-[arr:ARRIVED_AT]-(:Aircraft)
        WHERE arr.confidence IN ['HIGH', 'MEDIUM']
    WITH ap,
         count(DISTINCT dep) AS departures,
         count(DISTINCT arr) AS arrivals
    WHERE departures + arrivals > 0
    RETURN ap.icao    AS icao,
           ap.iata    AS iata,
           ap.name    AS name,
           ap.country AS country,
           departures,
           arrivals,
           departures - arrivals AS net_flow
    ORDER BY departures + arrivals DESC
    LIMIT 30
""")
print("Tráfico neto por aeropuerto (positivo = más salidas, negativo = más llegadas):")
print(df.to_string(index=False))

### Q11 – Historial completo de un avión: aeropuertos visitados

Reconstruye la secuencia de aeropuertos para el avión con más eventos.

In [ ]:
# Avión con más eventos de vuelo
with driver.session(database=NEO4J_DATABASE) as s:
    top = s.run("""
        MATCH (a:Aircraft)-[r:DEPARTED_FROM|ARRIVED_AT]->()
        RETURN a.icao24 AS icao24, count(r) AS n
        ORDER BY n DESC LIMIT 1
    """).single()
    if top:
        FLIGHT_ICAO = top["icao24"]
        print(f"Avión con más eventos: {FLIGHT_ICAO} ({top['n']} eventos)")
    else:
        FLIGHT_ICAO = EXAMPLE_ICAO
        print(f"Sin eventos aún — usando {FLIGHT_ICAO} de ejemplo")

df = run("""
    MATCH (a:Aircraft {icao24: $icao24})-[r:DEPARTED_FROM|ARRIVED_AT]->(ap:Airport)
    RETURN type(r)       AS event_type,
           r.event_time  AS event_time,
           ap.icao       AS airport_icao,
           ap.iata       AS airport_iata,
           ap.name       AS airport_name,
           ap.city       AS city,
           ap.country    AS country,
           r.confidence  AS confidence,
           r.dist_km     AS dist_km
    ORDER BY event_time ASC
""", icao24=FLIGHT_ICAO)
print(f"\nHistorial de {FLIGHT_ICAO}:")
print(df.to_string(index=False))

## 5. Cerrar conexión

In [ ]:
driver.close()
print("Conexión cerrada.")